# 14 Figure 3 Model Comparison and Descriptor Dominance

Purpose:
This notebook prepares draft plots and exported data for Figure 3.

Figure 3A:
- Tuned model-family comparison

Figure 3B:
- Model A vs Model B descriptor augmentation
- Model A: mechanistic / categorical baseline
- Model B: Model A + internal resistance descriptor

Figure 3C:
- Feature-block ablation
- Delta R² relative to the full model

Figure 3D:
- SHAP-based descriptor importance

The plots generated here are draft plots for inspection. Final figure panels will be redrawn in Igor Pro.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Output folder
out_dir = Path("../results/figure3_model_comparison")
out_dir.mkdir(parents=True, exist_ok=True)
r1c2_dir = Path("../results/revision/R1C2")
if not r1c2_dir.exists():
    raise FileNotFoundError(f"Missing validated R1C2 results: {r1c2_dir}")

print("Output folder:", out_dir.resolve())

# Optional global plot settings for draft plots
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.linewidth"] = 1.0
plt.rcParams["font.size"] = 10

Output folder: <REPOSITORY_ROOT>\results\figure3_model_comparison


In [2]:
# Okabe-Ito style colorblind-friendly palette
COLORS = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "green": "#009E73",
    "vermillion": "#D55E00",
    "purple": "#CC79A7",
    "sky": "#56B4E9",
    "yellow": "#F0E442",
    "black": "#000000",
    "gray": "#666666",
    "light_gray": "#DDDDDD"
}

## Part A — Figure 3A

In [3]:
# Validated R1C2 tuned model-family comparison (common observed-R subset).
fig3A_df = pd.read_csv(r1c2_dir / "model_family_common112.csv", float_precision="round_trip")
fig3A_df["analysis_subset"] = "common n=112 observed-R records"
fig3A_df["resistance_imputation"] = "none"

display(fig3A_df)

,model,descriptor_set,n_rows,cv_n_splits,cv_split_fingerprint,cv_r2_mean,cv_r2_std,analysis_subset,resistance_imputation
0,Elastic Net,Model_B_common112_observed_R,112,30,ee7a984b24b3b3d1fdef4d9ae97d5e4c4188789665b262...,0.202623,0.187157,common n=112 observed-R records,none
1,SVR,Model_B_common112_observed_R,112,30,ee7a984b24b3b3d1fdef4d9ae97d5e4c4188789665b262...,0.186745,0.185462,common n=112 observed-R records,none
2,Random Forest,Model_B_common112_observed_R,112,30,ee7a984b24b3b3d1fdef4d9ae97d5e4c4188789665b262...,0.178892,0.233892,common n=112 observed-R records,none
3,XGBoost,Model_B_common112_observed_R,112,30,ee7a984b24b3b3d1fdef4d9ae97d5e4c4188789665b262...,0.143494,0.231107,common n=112 observed-R records,none


In [4]:
# Figure 3A draft plot
fig, ax = plt.subplots(figsize=(5.2, 3.6))

x = np.arange(len(fig3A_df))

ax.errorbar(
    x,
    fig3A_df["cv_r2_mean"],
    yerr=fig3A_df["cv_r2_std"],
    fmt="o",
    markersize=6,
    capsize=4,
    color=COLORS["blue"],
    ecolor=COLORS["gray"],
    elinewidth=1.2
)

ax.axhline(0, linestyle="--", linewidth=1, color=COLORS["light_gray"])

ax.set_xticks(x)
ax.set_xticklabels(fig3A_df["model"], rotation=20, ha="right")
ax.set_ylabel("CV $R^2$")
ax.set_title("A  Tuned model-family comparison")
ax.set_ylim(-0.12, 0.45)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

<USER_HOME>\AppData\Local\Temp\ipykernel_3588\900315368.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Part B — Figure 3B

In [5]:
# Validated R1C2 Model A versus Model B comparison.
# Both descriptor sets use the same 112 records and identical CV splits.
fig3B_source = pd.read_csv(r1c2_dir / "modelA_modelB_common112.csv", float_precision="round_trip")
fig3B_df = fig3B_source.rename(columns={
    "model_A_cv_r2_mean": "model_A",
    "model_A_cv_r2_std": "model_A_std",
    "model_B_cv_r2_mean": "model_B",
    "model_B_cv_r2_std": "model_B_std",
    "delta_B_minus_A_mean": "delta_B_minus_A",
    "delta_B_minus_A_std": "delta_B_minus_A_std",
}).copy()
fig3B_df["analysis_subset"] = "common n=112 observed-R records; identical Model A/B splits"
fig3B_df["resistance_imputation"] = "none"

display(fig3B_df)

,model,n_rows,cv_n_splits,cv_split_fingerprint,model_A,model_A_std,model_B,model_B_std,delta_B_minus_A,delta_B_minus_A_std,analysis_subset,resistance_imputation
0,Elastic Net,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.136821,0.175933,0.243439,0.213699,0.106618,0.117541,common n=112 observed-R records; identical Mod...,none
1,SVR,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,-0.104000,0.351187,0.082710,0.325832,0.186711,0.143332,common n=112 observed-R records; identical Mod...,none
2,Random Forest,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.079085,0.302302,0.146954,0.273572,0.067869,0.202062,common n=112 observed-R records; identical Mod...,none
3,XGBoost,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,-0.108481,0.416516,0.021035,0.277275,0.129517,0.306342,common n=112 observed-R records; identical Mod...,none


In [6]:
# Figure 3B draft plot
fig, ax = plt.subplots(figsize=(5.2, 3.6))

x_positions = [0, 1]

model_colors = {
    "Elastic Net": COLORS["blue"],
    "Random Forest": COLORS["green"],
    "XGBoost": COLORS["orange"],
    "SVR": COLORS["purple"],
}

for row in fig3B_df.itertuples():
    ax.plot(
        x_positions,
        [row.model_A, row.model_B],
        marker="o",
        markersize=5,
        linewidth=1.5,
        color=model_colors[row.model],
        label=row.model
    )

ax.axhline(0, linestyle="--", linewidth=1, color=COLORS["light_gray"])

ax.set_xticks(x_positions)
ax.set_xticklabels(["Model A\nbaseline", "Model B\n+ log(R)"])
ax.set_ylabel("CV $R^2$")
ax.set_title("B  Descriptor augmentation")
ax.set_ylim(-0.12, 0.35)

ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1.0))

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

<USER_HOME>\AppData\Local\Temp\ipykernel_3588\3116478315.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Part C — Figure 3C

In [7]:
# Validated R1C2 feature-block ablation (common observed-R subset).
fig3C_source = pd.read_csv(r1c2_dir / "feature_block_ablation_common112.csv", float_precision="round_trip")
setting_labels = {
    "Mechanism labels": "no_mechanism",
    "Material class": "no_material",
    "Ion type": "no_ion_type",
    "Internal resistance": "no_internal_R",
    "Structure": "no_structure",
}
full_model_r2 = fig3C_source.loc[fig3C_source["removed_block"] == "Full model", "r2_mean"].iloc[0]
fig3C_df = fig3C_source.loc[fig3C_source["removed_block"] != "Full model"].copy()
fig3C_df = fig3C_df.rename(columns={"delta_vs_full_model": "delta_vs_baseline"})
fig3C_df["setting"] = fig3C_df["removed_block"].map(setting_labels)
fig3C_df["full_model_r2"] = full_model_r2
fig3C_df["analysis_subset"] = "common n=112 observed-R records"
fig3C_df["resistance_imputation"] = "none"

# Sort for lollipop plot
fig3C_plot_df = fig3C_df.sort_values("delta_vs_baseline", ascending=True).copy()

display(fig3C_df)

,removed_block,n_rows,cv_n_splits,cv_split_fingerprint,r2_mean,r2_std,delta_vs_baseline,setting,full_model_r2,analysis_subset,resistance_imputation
1,Internal resistance,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.086600,0.301229,-0.061813,no_internal_R,0.148413,common n=112 observed-R records,none
2,Structure,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.126776,0.274298,-0.021637,no_structure,0.148413,common n=112 observed-R records,none
3,Ion type,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.146080,0.269626,-0.002333,no_ion_type,0.148413,common n=112 observed-R records,none
4,Material class,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.094662,0.302150,-0.053751,no_material,0.148413,common n=112 observed-R records,none
5,Mechanism labels,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.151055,0.272526,0.002642,no_mechanism,0.148413,common n=112 observed-R records,none


In [8]:
# Figure 3C draft plot
fig, ax = plt.subplots(figsize=(5.2, 3.6))

y = np.arange(len(fig3C_plot_df))

for i, row in enumerate(fig3C_plot_df.itertuples()):
    color = COLORS["vermillion"] if row.delta_vs_baseline < 0 else COLORS["green"]
    
    ax.hlines(
        y=i,
        xmin=0,
        xmax=row.delta_vs_baseline,
        linewidth=1.5,
        color=color
    )
    ax.plot(
        row.delta_vs_baseline,
        i,
        "o",
        markersize=6,
        color=color
    )

ax.axvline(0, linestyle="--", linewidth=1, color=COLORS["gray"])

ax.set_yticks(y)
ax.set_yticklabels(fig3C_plot_df["removed_block"])
ax.set_xlabel(r"$\Delta$CV $R^2$ vs full model")
ax.set_title("C  Feature-block ablation")

ax.set_xlim(-0.10, 0.04)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

<USER_HOME>\AppData\Local\Temp\ipykernel_3588\2388116384.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Part D — Figure 3D

In [9]:
# Validated R1C2 SHAP descriptor-importance ranking (common observed-R subset).
fig3D_df = pd.read_csv(r1c2_dir / "shap_importance_common112.csv", float_precision="round_trip").rename(
    columns={"descriptor": "feature"}
).copy()
fig3D_df["analysis_subset"] = "common n=112 observed-R records"
fig3D_df["resistance_imputation"] = "none"

fig3D_plot_df = fig3D_df.nlargest(10, "mean_abs_shap").sort_values("mean_abs_shap", ascending=True).copy()

display(fig3D_df)

,rank,feature,encoded_feature,mean_abs_shap,n_rows_observed_R,holdout_random_state,analysis_subset,resistance_imputation
0,1,log(R),resistance__log_internal_resistance_Mohm,0.459942,112,42,common n=112 observed-R records,none
1,2,Other cation,cat__ion_type_other_cation,0.071902,112,42,common n=112 observed-R records,none
2,3,Top Electrode Cu,cat__top_electrode_Cu,0.051898,112,42,common n=112 observed-R records,none
3,4,Material Class Biomass,cat__material_class_biomass,0.050613,112,42,common n=112 observed-R records,none
4,5,Bottom Electrode Cu,cat__bottom_electrode_Cu,0.049784,112,42,common n=112 observed-R records,none
5,6,Film structure,cat__structure_class_film,0.049012,112,42,common n=112 observed-R records,none
6,7,Top Electrode Au,cat__top_electrode_Au,0.047250,112,42,common n=112 observed-R records,none
7,8,Proton,cat__ion_type_proton,0.046657,112,42,common n=112 observed-R records,none
8,9,Bottom Electrode Au,cat__bottom_electrode_Au,0.040717,112,42,common n=112 observed-R records,none
9,10,Material Class Polymer,cat__material_class_polymer,0.038177,112,42,common n=112 observed-R records,none


In [10]:
# Figure 3D draft plot
fig, ax = plt.subplots(figsize=(5.2, 3.6))

y = np.arange(len(fig3D_plot_df))

for i, row in enumerate(fig3D_plot_df.itertuples()):
    color = COLORS["blue"] if row.feature == "log(R)" else COLORS["gray"]
    
    ax.hlines(
        y=i,
        xmin=0,
        xmax=row.mean_abs_shap,
        linewidth=1.5,
        color=color
    )
    ax.plot(
        row.mean_abs_shap,
        i,
        "o",
        markersize=6,
        color=color
    )

ax.set_yticks(y)
ax.set_yticklabels(fig3D_plot_df["feature"])
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("D  SHAP descriptor importance")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

<USER_HOME>\AppData\Local\Temp\ipykernel_3588\2762893052.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Part E — Combined Figure 3 draft

In [11]:
# Combined 2 × 2 draft
fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.2))

# =========================
# A. Tuned model comparison
# =========================
ax = axes[0, 0]
x = np.arange(len(fig3A_df))

ax.errorbar(
    x,
    fig3A_df["cv_r2_mean"],
    yerr=fig3A_df["cv_r2_std"],
    fmt="o",
    markersize=6,
    capsize=4,
    color=COLORS["blue"],
    ecolor=COLORS["gray"],
    elinewidth=1.2
)

ax.axhline(0, linestyle="--", linewidth=1, color=COLORS["light_gray"])
ax.set_xticks(x)
ax.set_xticklabels(fig3A_df["model"], rotation=20, ha="right")
ax.set_ylabel("CV $R^2$")
ax.set_title("A  Tuned model-family comparison")
ax.set_ylim(-0.12, 0.45)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


# =========================
# B. Model A vs Model B
# =========================
ax = axes[0, 1]

for row in fig3B_df.itertuples():
    ax.plot(
        [0, 1],
        [row.model_A, row.model_B],
        marker="o",
        markersize=5,
        linewidth=1.5,
        color=model_colors[row.model],
        label=row.model
    )

ax.axhline(0, linestyle="--", linewidth=1, color=COLORS["light_gray"])
ax.set_xticks([0, 1])
ax.set_xticklabels(["Model A\nbaseline", "Model B\n+ log(R)"])
ax.set_ylabel("CV $R^2$")
ax.set_title("B  Descriptor augmentation")
ax.set_ylim(-0.12, 0.35)
ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1.0))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


# =========================
# C. Feature-block ablation
# =========================
ax = axes[1, 0]

y = np.arange(len(fig3C_plot_df))

for i, row in enumerate(fig3C_plot_df.itertuples()):
    color = COLORS["vermillion"] if row.delta_vs_baseline < 0 else COLORS["green"]
    ax.hlines(
        y=i,
        xmin=0,
        xmax=row.delta_vs_baseline,
        linewidth=1.5,
        color=color
    )
    ax.plot(
        row.delta_vs_baseline,
        i,
        "o",
        markersize=6,
        color=color
    )

ax.axvline(0, linestyle="--", linewidth=1, color=COLORS["gray"])
ax.set_yticks(y)
ax.set_yticklabels(fig3C_plot_df["removed_block"])
ax.set_xlabel(r"$\Delta$CV $R^2$ vs full model")
ax.set_title("C  Feature-block ablation")
ax.set_xlim(-0.10, 0.04)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


# =========================
# D. SHAP descriptor importance
# =========================
ax = axes[1, 1]

y = np.arange(len(fig3D_plot_df))

for i, row in enumerate(fig3D_plot_df.itertuples()):
    color = COLORS["blue"] if row.feature == "log(R)" else COLORS["gray"]
    ax.hlines(
        y=i,
        xmin=0,
        xmax=row.mean_abs_shap,
        linewidth=1.5,
        color=color
    )
    ax.plot(
        row.mean_abs_shap,
        i,
        "o",
        markersize=6,
        color=color
    )

ax.set_yticks(y)
ax.set_yticklabels(fig3D_plot_df["feature"])
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("D  SHAP descriptor importance")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# Save draft figure
fig.savefig(out_dir / "figure3_model_comparison_draft.png", dpi=300, bbox_inches="tight")
fig.savefig(out_dir / "figure3_model_comparison_draft.pdf", bbox_inches="tight")

plt.show()

<USER_HOME>\AppData\Local\Temp\ipykernel_3588\370324601.py:130: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Part F — Export CSV files for Igor Pro

In [12]:
# Export data. Read the validated CSVs as text here so their exact numeric
# representations are preserved in the production artifacts.
fig3A_export = pd.read_csv(r1c2_dir / "model_family_common112.csv", dtype=str, keep_default_na=False)
fig3A_export["analysis_subset"] = "common n=112 observed-R records"
fig3A_export["resistance_imputation"] = "none"

fig3B_export = pd.read_csv(r1c2_dir / "modelA_modelB_common112.csv", dtype=str, keep_default_na=False).rename(columns={
    "model_A_cv_r2_mean": "model_A",
    "model_A_cv_r2_std": "model_A_std",
    "model_B_cv_r2_mean": "model_B",
    "model_B_cv_r2_std": "model_B_std",
    "delta_B_minus_A_mean": "delta_B_minus_A",
    "delta_B_minus_A_std": "delta_B_minus_A_std",
})
fig3B_export["analysis_subset"] = "common n=112 observed-R records; identical Model A/B splits"
fig3B_export["resistance_imputation"] = "none"

fig3C_export_source = pd.read_csv(r1c2_dir / "feature_block_ablation_common112.csv", dtype=str, keep_default_na=False)
fig3C_export = fig3C_export_source.loc[fig3C_export_source["removed_block"] != "Full model"].copy()
fig3C_export = fig3C_export.rename(columns={"delta_vs_full_model": "delta_vs_baseline"})
fig3C_export["setting"] = fig3C_export["removed_block"].map(setting_labels)
fig3C_export["full_model_r2"] = fig3C_export_source.loc[fig3C_export_source["removed_block"] == "Full model", "r2_mean"].iloc[0]
fig3C_export["analysis_subset"] = "common n=112 observed-R records"
fig3C_export["resistance_imputation"] = "none"

fig3D_export = pd.read_csv(r1c2_dir / "shap_importance_common112.csv", dtype=str, keep_default_na=False).rename(columns={"descriptor": "feature"})
fig3D_export["analysis_subset"] = "common n=112 observed-R records"
fig3D_export["resistance_imputation"] = "none"

fig3A_export.to_csv(out_dir / "fig3A_tuned_model_comparison.csv", index=False, encoding="utf-8-sig")
fig3B_export.to_csv(out_dir / "fig3B_modelA_modelB_descriptor_augmentation.csv", index=False, encoding="utf-8-sig")
fig3C_export.to_csv(out_dir / "fig3C_feature_block_ablation.csv", index=False, encoding="utf-8-sig")
fig3D_export.to_csv(out_dir / "fig3D_SHAP_descriptor_importance.csv", index=False, encoding="utf-8-sig")

print("Exported files:")
for f in sorted(out_dir.glob("*")):
    print(f.name)

Exported files:
fig3A_tuned_model_comparison.csv
fig3B_modelA_modelB_descriptor_augmentation.csv
fig3C_feature_block_ablation.csv
fig3D_SHAP_descriptor_importance.csv
figure3_model_comparison_draft.pdf
figure3_model_comparison_draft.png
final_panels


## Figure 3 summary

Figure 3 summarizes model comparison and descriptor dominance.

- All Fig. 3 panels use the validated common n=112 observed-internal-resistance subset; internal resistance is not imputed.
- Fig. 3A shows the tuned model-family comparison for the revised Model B population.
- Fig. 3B shows Model A-to-Model B improvement on identical records and identical CV splits.
- Fig. 3C shows the internal-resistance and material-class blocks produce the largest performance losses on this common subset.
- Fig. 3D shows log(R) is the dominant individual descriptor according to SHAP importance.

Together, these results support the transition from categorical mechanism labels to transport-centered descriptor analysis.

# Final Figure 3 production

This section generates the final single-panel plots for Figure 3.

Generated panels:
- Fig. 3A: tuned model-family comparison
- Fig. 3B: descriptor augmentation from Model A to Model B
- Fig. 3C: feature-block ablation
- Fig. 3D: SHAP descriptor importance

The original analysis and draft plotting cells above are kept unchanged. Final multi-panel assembly is performed in Adobe Illustrator.

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# Output folders
# =========================
fig3_dir = Path("../results/figure3_model_comparison")
final_dir = fig3_dir / "final_panels"
final_dir.mkdir(parents=True, exist_ok=True)

print("Figure 3 result folder:", fig3_dir.resolve())
print("Final panel output folder:", final_dir.resolve())


# =========================
# Final style
# =========================
COLORS_FINAL = {
    "black": "#000000",
    "dark_gray": "#4D4D4D",
    "gray": "#8C8C8C",
    "light_gray": "#C7C7C7",
    "very_light_gray": "#EFEFEF",

    "blue": "#0072B2",
    "sky_blue": "#56B4E9",
    "green": "#009E73",
    "orange": "#E69F00",
    "vermillion": "#D55E00",
    "purple": "#CC79A7",
    "yellow": "#F0E442",
}

SEMANTIC_FINAL = {
    # Neutral plotting
    "neutral": "#8C8C8C",
    "neutral_light": "#BDBDBD",
    "neutral_fill": "#EFEFEF",

    # Key descriptor / improved model
    "descriptor": "#009E73",
    "descriptor_fill": "#BFD8CF",

    # Performance loss / important ablation
    "loss": "#D55E00",
    "loss_fill": "#E7D3BE",

    # Reference
    "reference": "#D0D0D0",
    "highlight": "#000000",
}


def set_final_style():
    mpl.rcParams.update({
        "font.family": "Arial",
        "font.size": 8.5,
        "font.weight": "normal",

        "mathtext.fontset": "custom",
        "mathtext.rm": "Arial",
        "mathtext.it": "Arial:italic",
        "mathtext.bf": "Arial:bold",

        "axes.linewidth": 1.0,
        "axes.labelsize": 9.5,
        "axes.titlesize": 10,
        "axes.labelweight": "normal",
        "axes.titleweight": "normal",

        "xtick.labelsize": 8.0,
        "ytick.labelsize": 8.0,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,

        "lines.linewidth": 1.2,
        "lines.markersize": 4,

        "legend.fontsize": 7.8,
        "legend.frameon": False,

        # Illustrator-friendly vector output
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",

        "savefig.dpi": 600,
        "figure.dpi": 150,
        "axes.grid": False,
    })


def format_final_ax(ax, mirror=True):
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
        spine.set_color(COLORS_FINAL["black"])

    ax.spines["top"].set_visible(mirror)
    ax.spines["right"].set_visible(mirror)

    ax.tick_params(
        direction="out",
        width=1.0,
        length=4,
        color=COLORS_FINAL["black"],
        top=False,
        right=False,
    )

    return ax


def save_final_panel(fig, name):
    pdf_path = final_dir / f"{name}.pdf"
    svg_path = final_dir / f"{name}.svg"
    png_path = final_dir / f"{name}.png"

    fig.savefig(pdf_path, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(svg_path, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.03)

    print("Saved:")
    print(" -", pdf_path.name)
    print(" -", svg_path.name)
    print(" -", png_path.name)


set_final_style()


# =========================
# Load exported Figure 3 data
# =========================
fig3A_path = fig3_dir / "fig3A_tuned_model_comparison.csv"
fig3B_path = fig3_dir / "fig3B_modelA_modelB_descriptor_augmentation.csv"
fig3C_path = fig3_dir / "fig3C_feature_block_ablation.csv"
fig3D_path = fig3_dir / "fig3D_SHAP_descriptor_importance.csv"

required_paths = [fig3A_path, fig3B_path, fig3C_path, fig3D_path]

for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}. Run the export cell above first.")

fig3A = pd.read_csv(fig3A_path, encoding="utf-8-sig")
fig3B = pd.read_csv(fig3B_path, encoding="utf-8-sig")
fig3C = pd.read_csv(fig3C_path, encoding="utf-8-sig")
fig3D = pd.read_csv(fig3D_path, encoding="utf-8-sig")

print("Loaded final plotting data:")
print(" -", fig3A_path.name, fig3A.shape)
print(" -", fig3B_path.name, fig3B.shape)
print(" -", fig3C_path.name, fig3C.shape)
print(" -", fig3D_path.name, fig3D.shape)

display(fig3A)
display(fig3B)
display(fig3C)
display(fig3D)

Figure 3 result folder: <REPOSITORY_ROOT>\results\figure3_model_comparison
Final panel output folder: <REPOSITORY_ROOT>\results\figure3_model_comparison\final_panels
Loaded final plotting data:
 - fig3A_tuned_model_comparison.csv (4, 9)
 - fig3B_modelA_modelB_descriptor_augmentation.csv (4, 12)
 - fig3C_feature_block_ablation.csv (5, 11)
 - fig3D_SHAP_descriptor_importance.csv (85, 8)


,model,descriptor_set,n_rows,cv_n_splits,cv_split_fingerprint,cv_r2_mean,cv_r2_std,analysis_subset,resistance_imputation
0,Elastic Net,Model_B_common112_observed_R,112,30,ee7a984b24b3b3d1fdef4d9ae97d5e4c4188789665b262...,0.202623,0.187157,common n=112 observed-R records,none
1,SVR,Model_B_common112_observed_R,112,30,ee7a984b24b3b3d1fdef4d9ae97d5e4c4188789665b262...,0.186745,0.185462,common n=112 observed-R records,none
2,Random Forest,Model_B_common112_observed_R,112,30,ee7a984b24b3b3d1fdef4d9ae97d5e4c4188789665b262...,0.178892,0.233892,common n=112 observed-R records,none
3,XGBoost,Model_B_common112_observed_R,112,30,ee7a984b24b3b3d1fdef4d9ae97d5e4c4188789665b262...,0.143494,0.231107,common n=112 observed-R records,none


,model,n_rows,cv_n_splits,cv_split_fingerprint,model_A,model_A_std,model_B,model_B_std,delta_B_minus_A,delta_B_minus_A_std,analysis_subset,resistance_imputation
0,Elastic Net,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.136821,0.175933,0.243439,0.213699,0.106618,0.117541,common n=112 observed-R records; identical Mod...,none
1,SVR,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,-0.104000,0.351187,0.082710,0.325832,0.186711,0.143332,common n=112 observed-R records; identical Mod...,none
2,Random Forest,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.079085,0.302302,0.146954,0.273572,0.067869,0.202062,common n=112 observed-R records; identical Mod...,none
3,XGBoost,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,-0.108481,0.416516,0.021035,0.277275,0.129517,0.306342,common n=112 observed-R records; identical Mod...,none


,removed_block,n_rows,cv_n_splits,cv_split_fingerprint,r2_mean,r2_std,delta_vs_baseline,setting,full_model_r2,analysis_subset,resistance_imputation
0,Internal resistance,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.086600,0.301229,-0.061813,no_internal_R,0.148413,common n=112 observed-R records,none
1,Structure,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.126776,0.274298,-0.021637,no_structure,0.148413,common n=112 observed-R records,none
2,Ion type,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.146080,0.269626,-0.002333,no_ion_type,0.148413,common n=112 observed-R records,none
3,Material class,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.094662,0.302150,-0.053751,no_material,0.148413,common n=112 observed-R records,none
4,Mechanism labels,112,50,99c6bc2d82d00a5732d570755ecb2a6258bdcca222d62c...,0.151055,0.272526,0.002642,no_mechanism,0.148413,common n=112 observed-R records,none


,rank,feature,encoded_feature,mean_abs_shap,n_rows_observed_R,holdout_random_state,analysis_subset,resistance_imputation
0,1,log(R),resistance__log_internal_resistance_Mohm,0.459942,112,42,common n=112 observed-R records,none
1,2,Other cation,cat__ion_type_other_cation,0.071902,112,42,common n=112 observed-R records,none
2,3,Top Electrode Cu,cat__top_electrode_Cu,0.051898,112,42,common n=112 observed-R records,none
3,4,Material Class Biomass,cat__material_class_biomass,0.050613,112,42,common n=112 observed-R records,none
4,5,Bottom Electrode Cu,cat__bottom_electrode_Cu,0.049784,112,42,common n=112 observed-R records,none
5,6,Film structure,cat__structure_class_film,0.049012,112,42,common n=112 observed-R records,none
6,7,Top Electrode Au,cat__top_electrode_Au,0.047250,112,42,common n=112 observed-R records,none
7,8,Proton,cat__ion_type_proton,0.046657,112,42,common n=112 observed-R records,none
8,9,Bottom Electrode Au,cat__bottom_electrode_Au,0.040717,112,42,common n=112 observed-R records,none
9,10,Material Class Polymer,cat__material_class_polymer,0.038177,112,42,common n=112 observed-R records,none


In [14]:
# =========================
# Final Fig. 3A
# Tuned model-family comparison
# =========================

set_final_style()

fig, ax = plt.subplots(figsize=(3.35, 2.65))

model_order = ["Elastic Net", "SVR", "Random Forest", "XGBoost"]

fig3A_plot = (
    fig3A
    .set_index("model")
    .loc[model_order]
    .reset_index()
)

x = np.arange(len(fig3A_plot))

ax.errorbar(
    x,
    fig3A_plot["cv_r2_mean"],
    yerr=fig3A_plot["cv_r2_std"],
    fmt="o",
    color=COLORS_FINAL["black"],
    ecolor=COLORS_FINAL["gray"],
    elinewidth=1.0,
    capsize=3,
    markersize=4.5,
    markerfacecolor=COLORS_FINAL["black"],
    markeredgecolor=COLORS_FINAL["black"],
    zorder=3,
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.0,
    color=SEMANTIC_FINAL["reference"],
    zorder=0,
)

ax.set_xticks(x)
ax.set_xticklabels(fig3A_plot["model"], rotation=25, ha="right")

ax.set_ylabel(r"$R^2_{\mathrm{CV}}$")
ax.set_title("Tuned model-family comparison")
ax.set_ylim(-0.12, 0.45)

format_final_ax(ax, mirror=True)

plt.tight_layout()

save_final_panel(fig, "fig3A_final_tuned_model_comparison")

plt.show()

Saved:
 - fig3A_final_tuned_model_comparison.pdf
 - fig3A_final_tuned_model_comparison.svg
 - fig3A_final_tuned_model_comparison.png


<USER_HOME>\AppData\Local\Temp\ipykernel_3588\3870733132.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# =========================
# Final Fig. 3B
# Descriptor augmentation
# =========================

set_final_style()

fig, ax = plt.subplots(figsize=(3.65, 2.65))

model_order = ["Elastic Net", "Random Forest", "XGBoost", "SVR"]

fig3B_plot = (
    fig3B
    .set_index("model")
    .loc[model_order]
    .reset_index()
)

y = np.arange(len(fig3B_plot))

for i, row in enumerate(fig3B_plot.itertuples()):
    # Connecting line
    ax.hlines(
        y=i,
        xmin=row.model_A,
        xmax=row.model_B,
        linewidth=1.4,
        color=COLORS_FINAL["gray"],
        zorder=1,
    )

    # Model A baseline
    ax.plot(
        row.model_A,
        i,
        "o",
        markersize=4.5,
        color=SEMANTIC_FINAL["neutral_light"],
        markeredgecolor=COLORS_FINAL["black"],
        markeredgewidth=0.4,
        zorder=2,
        label="Model A baseline" if i == 0 else None,
    )

    # Model B + log(R)
    ax.plot(
        row.model_B,
        i,
        "o",
        markersize=4.8,
        color=SEMANTIC_FINAL["descriptor"],
        markeredgecolor=COLORS_FINAL["black"],
        markeredgewidth=0.4,
        zorder=3,
        label=r"Model B + $\log(R)$" if i == 0 else None,
    )

ax.axvline(
    0,
    linestyle="--",
    linewidth=1.0,
    color=SEMANTIC_FINAL["reference"],
    zorder=0,
)

ax.set_yticks(y)
ax.set_yticklabels(fig3B_plot["model"])

ax.set_xlabel(r"$R^2_{\mathrm{CV}}$")
ax.set_title("Descriptor augmentation")
ax.set_xlim(-0.13, 0.32)

format_final_ax(ax, mirror=True)

ax.legend(
    loc="lower right",
    frameon=False,
    handlelength=1.2,
)

plt.tight_layout()

save_final_panel(fig, "fig3B_final_descriptor_augmentation")

plt.show()

Saved:
 - fig3B_final_descriptor_augmentation.pdf
 - fig3B_final_descriptor_augmentation.svg
 - fig3B_final_descriptor_augmentation.png


<USER_HOME>\AppData\Local\Temp\ipykernel_3588\2087225835.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# =========================
# Final Fig. 3C
# Feature-block ablation
# =========================

set_final_style()

fig, ax = plt.subplots(figsize=(3.85, 2.65))

fig3C_plot = (
    fig3C
    .sort_values("delta_vs_baseline", ascending=True)
    .copy()
)

y = np.arange(len(fig3C_plot))

for i, row in enumerate(fig3C_plot.itertuples()):
    if row.delta_vs_baseline < -0.005:
        color = SEMANTIC_FINAL["loss"]
        lw = 1.5
        ms = 4.8
    else:
        color = SEMANTIC_FINAL["neutral"]
        lw = 1.2
        ms = 4.2

    ax.hlines(
        y=i,
        xmin=0,
        xmax=row.delta_vs_baseline,
        linewidth=lw,
        color=color,
        zorder=2,
    )

    ax.plot(
        row.delta_vs_baseline,
        i,
        "o",
        markersize=ms,
        color=color,
        zorder=3,
    )

ax.axvline(
    0,
    linestyle="--",
    linewidth=1.0,
    color=COLORS_FINAL["gray"],
    zorder=0,
)

ax.set_yticks(y)
ax.set_yticklabels(fig3C_plot["removed_block"])

ax.set_xlabel(r"$\Delta R^2_{\mathrm{CV}}$ vs full model")
ax.set_title("Feature-block ablation")
ax.set_xlim(-0.095, 0.025)

format_final_ax(ax, mirror=True)

plt.tight_layout()

save_final_panel(fig, "fig3C_final_feature_block_ablation")

plt.show()

Saved:
 - fig3C_final_feature_block_ablation.pdf
 - fig3C_final_feature_block_ablation.svg
 - fig3C_final_feature_block_ablation.png


<USER_HOME>\AppData\Local\Temp\ipykernel_3588\3285378969.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# =========================
# Final Fig. 3D
# SHAP descriptor importance
# =========================

set_final_style()

fig, ax = plt.subplots(figsize=(3.95, 3.05))

fig3D_plot = (
    fig3D
    .nlargest(10, "mean_abs_shap")
    .sort_values("mean_abs_shap", ascending=True)
    .copy()
)

y = np.arange(len(fig3D_plot))

for i, row in enumerate(fig3D_plot.itertuples()):
    if row.feature == "log(R)":
        color = SEMANTIC_FINAL["descriptor"]
        lw = 1.9
        ms = 5.2
    else:
        color = SEMANTIC_FINAL["neutral"]
        lw = 1.15
        ms = 4.0

    ax.hlines(
        y=i,
        xmin=0,
        xmax=row.mean_abs_shap,
        linewidth=lw,
        color=color,
        zorder=2,
    )

    ax.plot(
        row.mean_abs_shap,
        i,
        "o",
        markersize=ms,
        color=color,
        zorder=3,
    )

ax.set_yticks(y)
ax.set_yticklabels(fig3D_plot["feature"])

ax.set_xlabel("Mean |SHAP value|")
ax.set_title("SHAP descriptor importance")
ax.set_xlim(0, 0.50)

format_final_ax(ax, mirror=True)

plt.tight_layout()

save_final_panel(fig, "fig3D_final_SHAP_descriptor_importance")

plt.show()

Saved:
 - fig3D_final_SHAP_descriptor_importance.pdf
 - fig3D_final_SHAP_descriptor_importance.svg
 - fig3D_final_SHAP_descriptor_importance.png


<USER_HOME>\AppData\Local\Temp\ipykernel_3588\2516727391.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
print("Final Figure 3 panel files:")

for p in sorted(final_dir.glob("fig3*_final*")):
    print(" -", p.name)

Final Figure 3 panel files:
 - fig3A_final_tuned_model_comparison.pdf
 - fig3A_final_tuned_model_comparison.png
 - fig3A_final_tuned_model_comparison.svg
 - fig3B_final_descriptor_augmentation.pdf
 - fig3B_final_descriptor_augmentation.png
 - fig3B_final_descriptor_augmentation.svg
 - fig3C_final_feature_block_ablation.pdf
 - fig3C_final_feature_block_ablation.png
 - fig3C_final_feature_block_ablation.svg
 - fig3D_final_SHAP_descriptor_importance.pdf
 - fig3D_final_SHAP_descriptor_importance.png
 - fig3D_final_SHAP_descriptor_importance.svg
